### Question 1

Dataset: research_abstracts.json. 

#### Question 1.1 = Load and denoise the dataset
Remove URLs, HTML, punctuation/numbers, extra whitespace, missing values, and duplicate cleaned abstracts. Convert text to lowercase. The DOI is kept only as a reference, not as a predictive feature.

In [2]:
import re
import html
from pathlib import Path
import pandas as pd

# The JSON has to be in the same folder as this notebook or read_json won't find it.
df = pd.read_json("research_abstracts.json")
print("Original shape:", df.shape)
print("Original labels:\n", df["Label"].value_counts())

def clean_text(text):
    # Turn things like &amp; back into normal characters first, then lowercase everything
    text = html.unescape(str(text)).lower()
    # Links don't say anything about the topic, so they go
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    # Some abstracts were scraped from web pages and still have HTML tags in them
    text = re.sub(r"<[^>]+>", " ", text)
    # Keep only letters and spaces - this gets rid of punctuation and numbers in one go
    text = re.sub(r"[^a-z\s]", " ", text)
    # Squash the extra spaces left behind by the replacements above
    return re.sub(r"\s+", " ", text).strip()

# Keep the columns I need and drop rows with no abstract or no label
df = df[["DOI", "Abstract", "Label"]].dropna(subset=["Abstract", "Label"]).copy()
# Just in case, make sure only the two expected labels are left
df = df[df["Label"].isin(["ai", "classical"])]
df["clean_text"] = df["Abstract"].apply(clean_text)
# Some abstracts end up empty or identical after cleaning, so remove those as well
df = df[df["clean_text"].ne("")].drop_duplicates(subset="clean_text").reset_index(drop=True)

print("Cleaned shape:", df.shape)
print("Cleaned labels:\n", df["Label"].value_counts())
# Quick look at a few cleaned rows (cut to 100 characters so the table stays readable)
display(df[["clean_text", "Label"]].head(3).assign(
    clean_text=lambda x: x["clean_text"].str.slice(0, 100) + "..."
))

Original shape: (1138, 3)
Original labels:
 Label
classical    900
ai           238
Name: count, dtype: int64
Cleaned shape: (1121, 4)
Cleaned labels:
 Label
classical    886
ai           235
Name: count, dtype: int64


,clean_text,Label
0,forward looking forecasting of the inflation r...,ai
1,our study examined the disaggregation of infla...,ai
2,this paper develops a dynamic factor model tha...,classical


#### Question 1.2 = Identify features and apply TF-IDF 

TF-IDF converts research vocabulary into numerical features. English stop words are removed by the vectorizer. Fit it on training data only to prevent test data leakage.

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# The cleaned abstract is the input, the label is what we want to predict
X = df["clean_text"]
y = df["Label"]
# 80/20 split. stratify keeps the ai/classical ratio the same in both sets,
# which matters because there are far more classical abstracts 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

# This is only here to show what the TF-IDF features look like. The actual model below
# fits its own TF-IDF inside a pipeline, so it only ever learns from the training data
tfidf_example = TfidfVectorizer(stop_words="english", max_features=6000)
X_train_tfidf = tfidf_example.fit_transform(X_train)
print("Training feature matrix:", X_train_tfidf.shape)
print("Example features:", tfidf_example.get_feature_names_out()[:15].tolist())
print("Training records:", len(X_train), "| Test records:", len(X_test))

Training feature matrix: (896, 6000)
Example features: ['aa', 'aaa', 'abandoned', 'abar', 'abct', 'abilities', 'ability', 'able', 'abm', 'abnormal', 'abo', 'abroad', 'abrupt', 'absence', 'absent']
Training records: 896 | Test records: 225


#### Question 1.3 = Train and evaluate the baseline 
Use a Multinomial Naive Bayes classifier with TF-IDF features. The pipeline fits TF-IDF only on the training split. Compare training and test macro-F1 and report precision, recall, F1 and accuracy on the test set.

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Putting TF-IDF and Naive Bayes in one pipeline means the vectoriser is fitted on
# X_train only, so nothing from the test set leaks into training
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english", max_features=6000)),
    ("model", MultinomialNB())
])
baseline.fit(X_train, y_train)

# Predict on both sets so I can compare how well it learned vs how well it generalises
base_train_pred = baseline.predict(X_train)
base_test_pred = baseline.predict(X_test)

# Macro-F1 weighs both classes equally. Accuracy on its own looks fine here just because
# the model can guess 'classical' most of the time and still be right
print("Baseline training macro-F1:", round(f1_score(y_train, base_train_pred, average="macro"), 3))
print("Baseline test macro-F1:", round(f1_score(y_test, base_test_pred, average="macro"), 3))
print("Baseline test accuracy:", round(accuracy_score(y_test, base_test_pred), 3))
print("\nBaseline classification report:\n", classification_report(y_test, base_test_pred, zero_division=0))
# Rows are the true labels, columns are what the model predicted
print("Confusion matrix [ai, classical]:\n", confusion_matrix(
    y_test, base_test_pred, labels=["ai", "classical"]
))

Baseline training macro-F1: 0.612
Baseline test macro-F1: 0.525
Baseline test accuracy: 0.809

Baseline classification report:
               precision    recall  f1-score   support

          ai       1.00      0.09      0.16        47
   classical       0.81      1.00      0.89       178

    accuracy                           0.81       225
   macro avg       0.90      0.54      0.52       225
weighted avg       0.85      0.81      0.74       225

Confusion matrix [ai, classical]:
 [[  4  43]
 [  0 178]]


**Baseline comment.** Accuracy is **80.9%**, but AI recall is only **8.5%** (43 of 47 AI abstracts are missed). Test macro-F1 is **0.525**, below training macro-F1 **0.612**. The dataset is imbalanced, and the baseline does not identify AI abstracts reliably.